<a href="https://colab.research.google.com/github/RajputChirag27/Car_Dekho_Car_Prices_Prediction/blob/main/car_prices_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [80]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import GridSearchCV

df = pd.read_csv("car_data.csv")
print(df.head())

                       name  year  selling_price  km_driven    fuel  \
0             Maruti 800 AC  2007          60000      70000  Petrol   
1  Maruti Wagon R LXI Minor  2007         135000      50000  Petrol   
2      Hyundai Verna 1.6 SX  2012         600000     100000  Diesel   
3    Datsun RediGO T Option  2017         250000      46000  Petrol   
4     Honda Amaze VX i-DTEC  2014         450000     141000  Diesel   

  seller_type transmission         owner  
0  Individual       Manual   First Owner  
1  Individual       Manual   First Owner  
2  Individual       Manual   First Owner  
3  Individual       Manual   First Owner  
4  Individual       Manual  Second Owner  


In [74]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4340 entries, 0 to 4339
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   name           4340 non-null   object
 1   year           4340 non-null   int64 
 2   selling_price  4340 non-null   int64 
 3   km_driven      4340 non-null   int64 
 4   fuel           4340 non-null   object
 5   seller_type    4340 non-null   object
 6   transmission   4340 non-null   object
 7   owner          4340 non-null   object
dtypes: int64(3), object(5)
memory usage: 271.4+ KB
None


In [75]:
df['car_age'] = 2024 - df['year']
df = df.drop(['name', 'year'], axis=1)
df['km_driven'] = np.log1p(df['km_driven'])
df['selling_price'] = np.log1p(df['selling_price'])

In [76]:
numerical_features = ['km_driven', 'car_age']
catergorical_features = ['fuel', 'seller_type', 'transmission', 'owner']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'),catergorical_features)
    ])

In [77]:
X = df.drop(['selling_price'], axis=1)
y = df['selling_price']

model_pipeline = Pipeline(
    steps=[('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=8))]
)

X_train,X_test,y_train,y_test = train_test_split(X,y, test_size=0.2, random_state=8)

In [70]:
param_grid = {
'regressor__n_estimators': [100, 200, 300],
'regressor__max_depth': [None, 10, 20, 30],
'regressor__min_samples_split': [2, 5, 10]}

grid_search = GridSearchCV(model_pipeline,param_grid,cv=5, scoring='r2',n_jobs=1)

grid_search.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         StandardScaler(),
                                                                         ['km_driven',
                                                                          'car_age']),
                                                                        ('cat',
                                                                         OneHotEncoder(handle_unknown='ignore'),
                                                                         ['fuel',
                                                                          'seller_type',
                                                                          'transmission',
                                                                          'owner'])])),
                                       ('regressor',
                                        RandomForestRegressor(random_state=8))]),
             n_jobs=1,
             param_grid={'regressor__max_depth': [None, 10, 20, 30],
                         'regressor__min_samples_split': [2, 5, 10],
                         'regressor__n_estimators': [100, 200, 300]},
             scoring='r2')

In [71]:
print("Best parameters", grid_search.best_params_)
print("Best score", grid_search.best_score_)
print("Best estimator", grid_search.best_estimator_)
print("Best index", grid_search.best_index_)
print("Best params", grid_search.best_params_)

Best parameters {'regressor__max_depth': 10, 'regressor__min_samples_split': 2, 'regressor__n_estimators': 300}
Best score 0.724176665001718
Best estimator Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['km_driven', 'car_age']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['fuel', 'seller_type',
                                                   'transmission',
                                                   'owner'])])),
                ('regressor',
                 RandomForestRegressor(max_depth=10, n_estimators=300,
                                       random_state=8))])
Best index 11
Best params {'regressor__max_depth': 10, 'regressor__min_samples_split': 2, 'regressor__n_estimators': 300}


In [81]:
best_model = grid_search.best_estimator_

y_pred_log = best_model.predict(X_test)

y_test_actual = np.expm1(y_test)
y_pred_actual = np.expm1(y_pred_log)

mse = mean_squared_error(y_test_actual, y_pred_actual)
r2 = r2_score(y_test_actual, y_pred_actual)
mae = mean_absolute_error(y_test_actual,y_pred_actual)

print("Mean Squared Error:", mse)
print("R-squared:", r2)
print("Root Mean Squared Error", np.sqrt(mse))
print("Mean Absolute Error",mae)


Mean Squared Error: 181010442157.31378
R-squared: 0.5488999921296394
Root Mean Squared Error 425453.2197049563
Mean Absolute Error 164150.8810986724


In [79]:
# import joblib
# import boto3
# import tempfile
# import os

# # 1. Model ko Disk par save karein (Serializing) [cite: 305, 312]
# # Hum 'grid_search' object ko save kar rahe hain jisme best_estimator_ shamil hai
# model_filename = 'grid_search.pkl'
# joblib.dump(grid_search, model_filename)
# print(f"Model saved to disk as {model_filename}")

# # 2. S3 Client initialize karein [cite: 306]
# s3 = boto3.client('s3')

# # 3. Model ko S3 bucket mein upload karein [cite: 308, 309, 313]
# # Bucket name 'car-data' challenge mein specified hai
# bucket_name = 'car-data'

# try:
#     s3.upload_file(model_filename, bucket_name, 'grid_search.pkl')
#     print(f"Successfully uploaded {model_filename} to S3 bucket: {bucket_name}")
# except Exception as e:
#     print(f"Error uploading to S3: {e}")
#     # Note: Exam environment mein permissions pehle se set hoti hain

results_df = pd.DataFrame({
    'Actual_Price': y_test_actual,
    'Predicted_Price': y_pred_actual,
    'Error': y_test_actual - y_pred_actual
})

# Is file ko disk par save karein
results_df.to_csv('model_predictions_output.csv', index=False)
print("✅ Predictions saved to 'model_predictions_output.csv'")

✅ Predictions saved to 'model_predictions_output.csv'
